<img src="images/logodwengo.png" alt="LogoDwengo" width="150"/>

<div>
    <font color=#690027 markdown="1">
        <h1>SIMULER UNE ÉPIDÉMIE: LE MODÈLE SIR</h1>    </font>
</div>

<div class="alert alert-box alert-success">
Dans ce projet, tu étudies comment des maladies peuvent se propager à travers un réseau (social). Tu examines comment la structure d’un réseau peut influencer la vitesse à laquelle une maladie est transmise. Finalement, tu examineras également différentes stratégies pour lutter contre la propagation d’une maladie.<br>Dans ce notebook, vous ferez connaissance avec le modèle mathématique SIR.</div>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.spatial import distance_matrix

## Le modèle SIR
L'une des manières les plus simples de modéliser la propagation d'une maladie dans une communauté est d'utiliser le modèle SIR. **SIR signifie *Susceptible* (vulnérable), *Infected* (infecté) et *Resistant* (résistant ou rétabli), les trois types d'individus présents dans une communauté.** <br>Le modèle SIR se compose de trois équations qui décrivent les variations du nombre d’individus dans un groupe donné. Les variables décrivant l’état sont :
-  $S(t)$: le nombre d'individus susceptibles au temps $t$;-  $I(t)$: le nombre d'individus infectés au temps $t$;-  $R(t)$: le nombre d'individus résistants à l'instant $t$.
Ici, t est le temps dans une certaine unité de temps (l’unité de temps est choisie en fonction du problème).
Cette description constitue une première grande simplification de la réalité. On suppose que chacune de ces variables est un nombre réel, et que le nombre d’individus dans chaque groupe peut varier de manière continue. En réalité, il s’agit de valeurs discrètes : le nombre d’infectés et d’individus susceptibles est un nombre naturel, puisque l’on est infecté ou on ne l’est pas. Cependant, les modélisateurs préfèrent travailler avec des variables continues, car ils peuvent ainsi utiliser les techniques de l’analyse mathématique.

> **Exercice 1**: Dans quelles conditions cette approximation continue est-elle à peu près valable ? Pensez-vous que ce modèle puisse être utilisé pour décrire une famille de quatre personnes ?

Réponse:

Ces trois variables sont reliées entre elles au moyen de trois équations différentielles (qui décrivent chacune une variation au cours du temps). On suppose que la taille de la population reste inchangée : on considère donc que, pendant l’intervalle de temps décrit par le modèle, personne ne naît et personne ne meurt. En fait, on se limite ici à la propagation d’une maladie relativement bénigne comme un rhume. On peut donc représenter la situation par le système d’équations différentielles suivant : 
$$\left\{    \begin{array}{31}        \Large\frac{\text{d}S(t)}{\text{d}t} \normalsize = -\beta \, S(t) \, I(t) \\        \Large\frac{\text{d}I(t)}{\text{d}t} \normalsize = \beta \, S(t) \,I(t) - \gamma \, I(t) \\        \Large\frac{\text{d}R(t)}{\text{d}t} \normalsize = \gamma \, I(t)    \end{array}\right.$$

<div class="alert alert-box alert-info">
Chaque équation indique comment le nombre de personnes dans chaque groupe évolue au cours du temps. On peut en déduire combien de personnes se trouvent à un moment donné dans chaque groupe. Les paramètres $\beta$ et $\gamma$ y jouent un rôle fondamental.</div>

Les équations sont couplées via les *pourcentages de transition* (voir figure). Chaque pourcentage de transition indique comment on passe d’un groupe à l’autre. <br>Le taux de transition de susceptible (S) à infecté (I) dépend du contact entre une personne susceptible et une personne infectée. On appelle cela le *taux d'infection* $\beta$. Cela signifie qu'une personne infectée infectera $\beta S$ personnes. Le nombre de personnes susceptibles diminue donc de $\beta S I$ par unité de temps. <br>Le taux de passage de l'état infecté (I) à l'état résistant (R) ne dépend que du *taux de guérison*, que l'on note $\gamma$. Le nombre de personnes infectées diminue donc de $\gamma I$ par unité de temps.<img src="images/overgangSIR.png" alt="overgang in SIR" width="400"/>
<center>Figure 1 : Transition d'un groupe à l'autre au sein du modèle SIR.</center>

> **Exercice 2**: Écris comment les effectifs au sein de chaque groupe évoluent par unité de temps.

Réponse:

<div class="alert alert-box alert-info">
Le modèle SIR est difficile à résoudre exactement. C'est le cas pour de nombreuses équations différentielles qui apparaissent dans les sciences biologiques. Il faut donc trouver une <em>approximation numérique</em> de la solution. Cela signifie que vous utiliserez un algorithme pour trouver une solution approchée mais précise. À partir de cette solution, vous pouvez comprendre comment les différentes variables évoluent au cours du temps.</div>

Il existe différentes possibilités pour le faire :
-  Vous pourriez remplacer le problème continu par un équivalent **discret**. <br>Cela vous permettrait d'utiliser certaines méthodes numériques pour obtenir une solution approchée.
-  D'autre part, vous pouvez utiliser une méthode **itérative**. <br>En partant d'une estimation initiale, les méthodes itératives effectuent des approximations successives qui convergent progressivement vers la solution exacte.

## Méthode itérative
Grâce aux ordinateurs, il est facile de trouver de manière itérative une solution numérique du modèle SIR.
- Pour ce faire, vous partez d'une *condition initiale* : il est logique de commencer avec une population comptant zéro personne résistante, quelques personnes infectées et le reste de la population susceptible (voir les exemples).- Ensuite, vous pouvez utiliser la solution numérique pour calculer le nombre de personnes dans chaque groupe à des instants donnés.
Grâce au module Python SciPy, vous pouvez facilement *simuler* de telles équations différentielles.- D'abord, il faut *implémenter* les équations différentielles : pour cela, on place les trois équations données ci-dessus dans une *matrice ligne*.<br> À l'aide du module Python NumPy, on peut définir une matrice avec un *tableau NumPy*.

In [ ]:
# ingeven differentiaalvergelijkingen
def SIR(t, y, beta, gamma):
    """Differentiaalvergelijkingen die S, I en R in functie van de tijd t bepalen."""
    S, I, R = y
    return np.array([-beta * S * I,
                    beta * S * I - gamma * I,
                    gamma * I])

- Maintenant, vous pouvez *résoudre numériquement* le système d'équations différentielles avec la fonction `solve_ivp()` du module SciPy pour une *condition initiale* donnée.

### Exemple 1Considérons une population de 1000 personnes, dont initialement une personne ($I_0$) est infectée et $S_0=999$ personnes sont susceptibles d’être infectées par la maladie.<br>Vous indiquez également les taux de transition : $ \beta = 0,001$ et $\gamma = 0,1$.

In [ ]:
# voorbeeld 1 
# beginsituatie
S0 = 999
I0 = 1
R0 = 0

beta = 0.001
gamma = 0.1

oplossing = solve_ivp(SIR,                     # functie met parameters
                      [0, 100],                # tijdsinterval waarin je simuleert
                      np.array([S0, I0, R0]),  # initiële omstandigheden
                      args=(beta, gamma))      # parameters van stelsel differentiaalvergelijkingen 

In [ ]:
print(oplossing)       # oplossing geeft rij t-waarden en matrix y met als rijen S, I en R terug  

Vous représentez ensuite cette solution graphiquement de différentes manières :

In [ ]:
# voorbeeld 1 grafiek oplossing S, I, R 
plt.figure()

plt.plot(oplossing.t, oplossing.y[0], color="orange")  # S
plt.plot(oplossing.t, oplossing.y[1], color="purple")  # I 
plt.plot(oplossing.t, oplossing.y[2], color="green")   # R

plt.show()

In [ ]:
# voorbeeld 1 grafiek verdeling populatie over S, I, R in functie van de tijd
plt.figure()

plt.stackplot(oplossing.t, oplossing.y[[1,0,2],:],
              labels=["I", "S", "R"],
              colors=["red", "yellow", "lightgreen"])
plt.xlabel("Tijd")
plt.ylabel("Aantal personen")
plt.legend(loc=0)

plt.show()

In [ ]:
# grafiek voorbeeld 1 combinatie verdeling populatie en S, I, R
plt.figure()

plt.stackplot(oplossing.t, oplossing.y[[1,0,2],:],
              labels=["I", "S", "R"],
              colors=["red", "yellow", "lightgreen"])
plt.xlabel("Tijd")
plt.ylabel("Aantal personen")
plt.legend(loc=0)

plt.plot(oplossing.t, oplossing.y[1], color="purple")  # I 
plt.plot(oplossing.t, oplossing.y[2], color="green")   # R
plt.plot(oplossing.t, oplossing.y[0], color="orange")  # S

plt.show()

> **Exercice 3**: Adaptez l'exemple 1 et simulez plusieurs situations en ajustant les paramètres :-  Que se passerait-il si, initialement, la moitié de la population était résistante ?-  Et si initialement 80 % de la population était résistante ?

### Exemple 2Même problème que dans l'exemple 2 mais avec d'autres pourcentages de transition.

In [ ]:
# voorbeeld 2 
# beginsituatie
S0 = 999
I0 = 1
R0 = 0

beta = 0.0001
gamma = 0.048


oplossing4 = solve_ivp(SIR,                    # functie met parameters
                      [0, 365],                # tijdsinterval waarin we simuleren
                      np.array([S0, I0, R0]),  # initiële omstandigheden
                      t_eval=np.linspace(0,365,36),   # aantal punten van oplossing
                      args=(beta, gamma))      # parameters van stelsel differentiaalvergelijkingen 

In [ ]:
# voorbeeld 2 grafiek oplossing S, I, R 
plt.figure()

plt.plot(oplossing4.t, oplossing4.y[0], color="orange")  # S
plt.plot(oplossing4.t, oplossing4.y[1], color="purple")  # I 
plt.plot(oplossing4.t, oplossing4.y[2], color="green")   # R

plt.show()

In [ ]:
# grafiek voorbeeld 2 grafiek verdeling populatie over S, I, R in functie van de tijd
plt.figure()

plt.stackplot(oplossing4.t, oplossing4.y[[1,0,2],:],
              labels=["I", "S", "R"],
              colors=["red", "yellow", "green"])
plt.xlabel("Tijd")
plt.ylabel("Aantal personen")
plt.legend(loc=0)

plt.show()

In [ ]:
# grafiek voorbeeld 2 combinatie verdeling populatie en S, I, R
plt.figure()

plt.stackplot(oplossing4.t, oplossing4.y[[1,0,2],:],
              labels=["I", "S", "R"],
              colors=["red", "yellow", "lightgreen"])
plt.xlabel("Tijd")
plt.ylabel("Aantal personen")
plt.legend(loc=0)

plt.plot(oplossing.t, oplossing.y[1], color="purple")  # I 
plt.plot(oplossing.t, oplossing.y[2], color="green")   # R
plt.plot(oplossing.t, oplossing.y[0], color="orange")  # S

plt.show()

> **Exercice 4**: Calculez la valeur de $S$ à l'instant où $I$ est maximal.

Réponse:

> **Exercice 5**: Modifiez l'exemple 2 en réduisant le taux d'infection $\beta$ d'un quart. Comment le graphique change-t-il ?

<img src="images/cclic.png" alt="Banner" align="left" width="100"/><br><br>
Ce notebook de M. Stock et F. wyffels pour Dwengo asbl est concédé sous une <a href="http://creativecommons.org/licenses/by-nc-sa/4.0/">licence Creative Commons Attribution - Pas d’Utilisation Commerciale - Partage dans les Mêmes Conditions 4.0 International</a>.